## Reference shorelines

Builds MHW, MLW and midline reference shorelines from OS OpenMap Local
`TidalBoundary`, trimmed to the open coast between Kilnsea and Bridlington.

**Method.** OS `TidalBoundary` features are merged per classification and the
longest connected component taken — this is the open coast plus the Humber north
bank. It is oriented south-to-north, then trimmed by linear referencing between
two anchor points. Rectangle clipping cannot separate the Humber from the open
coast because they overlap in both easting and northing.

All logic lives in `holderness/reference.py`. This notebook orchestrates, plots
and inspects. Parameters come from `holderness/config.py`.

**Outputs:** three Nx2 arrays in EPSG:27700 for use as
`settings['reference_shoreline']`, an intertidal separation profile, a metadata
record, and the WGS84 polygon for CoastSat image retrieval.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from holderness import config, reference

config.OUT.mkdir(parents=True, exist_ok=True)

config.FIGS.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    """Save a figure to outputs/figures with consistent settings."""
    for ext in ('png', 'pdf'):
        fig.savefig(config.FIGS / f'{name}.{ext}', dpi=config.FIG_DPI,
                    bbox_inches='tight')
        
print('repo :', config.REPO_ROOT)
print('gml  :', config.GML_PATH, '(exists)' if config.GML_PATH.exists() else '(MISSING)')
print('out  :', config.OUT)

## Load

In [ ]:
gdf_mhw, gdf_mlw, counts = reference.load_tidal_boundary(
    config.GML_PATH, config.LAYER, config.EPSG
)

for name, c in counts.items():
    print(f"{name}: {c['n_in']} features -> {c['n_out']} kept "
          f"(invalid {c['invalid']}, empty {c['empty']}, "
          f"null {c['null']}, non-finite {c['non_finite']})")

## Merge, orient south to north, trim

In [ ]:
# Component length before trimming shows how much was Humber and Hull valley

lines, comps = {}, {}
for name, g in [('mhw', gdf_mhw), ('mlw', gdf_mlw)]:
    comps[name], lines[name] = reference.build_reference_lines(
        g, config.ANCHOR_SOUTH, config.ANCHOR_NORTH
    )
    print(f'{name}: component {comps[name].length:8.0f} m '
          f'-> trimmed {lines[name].length:8.0f} m')

In [ ]:
# Regression check - change indicates source data or anchor points have changed

for name, line in lines.items():
    exp = config.EXPECT_LENGTH_M[name]
    rel = abs(line.length - exp) / exp
    print(f'{name}: {line.length:.0f} m vs expected {exp} m '
          f'({rel:.2%}) {"ok" if rel < config.EXPECT_TOL else "CHANGED"}')

## Hairpin diagnostic

In [ ]:
# OS traces up and back along drain and outfall channels crossing the foreshore.
# These are genuine mapped geometry, not merge artefacts, and are recorded rather than removed
# See the docstring in reference.hairpin_locations for why neither simplification nor smoothing is used

hairpins = {}
for name, line in lines.items():
    hairpins[name] = reference.hairpin_locations(line)
    print(f'{name}: {len(hairpins[name])} hairpins')
    for a, b in hairpins[name]:
        p = line.interpolate((a + b) / 2)
        print(f'   {a:6.0f}-{b:6.0f} m  at ({p.x:.0f}, {p.y:.0f})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, (name, line) in zip(axes, lines.items()):
    d, r = reference.hairpin_ratio(line)
    ax.plot(d / 1000, r, lw=0.7)
    ax.axhline(0.5, color='C3', ls='--', lw=0.8)
    ax.set_ylabel(f'{name} ratio')
    ax.set_ylim(0, 1.1)
    ax.grid(ls=':', color='0.8')
axes[-1].set_xlabel('alongshore distance from Kilnsea (km)')
fig.tight_layout()
savefig(fig, 'ref_hairpin_ratio')

## Inspect

In [ ]:
coords = {k: np.array(v.coords) for k, v in lines.items()}

fig, ax = plt.subplots(figsize=(7, 11))
ax.plot(*coords['mhw'].T, color='C1', lw=1, label='MHW')
ax.plot(*coords['mlw'].T, color='C0', lw=1, label='MLW')
ax.set_aspect('equal')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.legend()
ax.set_title(f'{config.SITE}: Kilnsea to Bridlington')
savefig(fig, 'ref_overview')

In [ ]:
def zoom_to(ax, along_m, half=500, line=None):
    """Zoom an axis to a point a given distance along a line (default MHW)."""
    line = lines['mhw'] if line is None else line
    p = line.interpolate(along_m)
    ax.set_xlim(p.x - half, p.x + half)
    ax.set_ylim(p.y - half, p.y + half)
    ax.figure.canvas.draw()
    return p


# MLW should sit consistently seaward (east) of MHW at every position
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, along in zip(axes, [5_000, 20_000, 35_000, 50_000]):
    ax.plot(*coords['mhw'].T, color='C1', lw=1)
    ax.plot(*coords['mlw'].T, color='C0', lw=1)
    ax.set_aspect('equal')
    ax.set_title(f'{along/1000:.0f} km')
    zoom_to(ax, along)
fig.tight_layout()
savefig(fig, 'ref_zoom_panels')

## Intertidal separation

In [ ]:
# Distance from each MHW sample to the nearest point on MLW. Sizes the tidal
# component of max_dist_ref, and cross-checks the beach slopes that SDS_slope
# will estimate later from an entirely different source

d_along, pts_mhw, sep = reference.separation_profile(
    lines['mhw'], lines['mlw'], config.SAMPLE_STEP
)

p5, p50, p95 = np.percentile(sep, [5, 50, 95])
print(f'separation (m): min {sep.min():.0f}  p5 {p5:.0f}  median {p50:.0f}  '
      f'p95 {p95:.0f}  max {sep.max():.0f}')

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(d_along / 1000, sep, lw=0.7)
ax.set_xlabel('alongshore distance from Kilnsea (km)')
ax.set_ylabel('MHW-MLW separation (m)')
ax.grid(ls=':', color='0.8')
fig.tight_layout()
savefig(fig, 'ref_separation_profile')

## Midline

In [ ]:
# Centring the reference on the tidal envelope rather than on MHW lets a tighter
# max_dist_ref cover the same range of waterline positions.

lines['midline'], ratio = reference.build_midline(pts_mhw, lines['mlw'])

print(f'midline length {lines["midline"].length:.0f} m')
print(f'mid-to-MHW / (separation/2): median {np.nanmedian(ratio):.3f}  '
      f'5-95pct {np.nanpercentile(ratio, 5):.3f}-{np.nanpercentile(ratio, 95):.3f}')

In [ ]:
mid = np.array(lines['midline'].coords)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*coords['mhw'].T, color='C1', lw=1, label='MHW')
ax.plot(*coords['mlw'].T, color='C0', lw=1, label='MLW')
ax.plot(*mid.T, color='C2', lw=1, label='midline')
ax.set_aspect('equal')
ax.legend(fontsize=8)
zoom_to(ax, 30_000, half=800)
savefig(fig, 'ref_midline_detail')

## Densify and save

In [ ]:
# CoastSat will build a distance buffer around the reference shoreline, so vertex
# spacing must be well below the intended max_dist_ref

arrays = {}
for name, line in lines.items():
    arrays[name] = reference.densify(line, config.DENSIFY_STEP)
    spacing = np.linalg.norm(np.diff(arrays[name], axis=0), axis=1)
    print(f'{name}: {len(arrays[name])} vertices, '
          f'spacing {spacing.min():.2f}-{spacing.max():.2f} m')

In [ ]:
for name, arr in arrays.items():
    fn = config.OUT / f'refsl_{config.SITE}_os_{name}.pkl'
    reference.save_reference(arr, fn)
    print('wrote', fn.name)

np.savez(config.OUT / f'separation_{config.SITE}.npz', along=d_along, separation=sep)
print('wrote', f'separation_{config.SITE}.npz')

In [ ]:
# Metadata alongside the arrays — a bare Nx2 array carries no CRS or provenance, so this is where that lives

meta = {
    'site': config.SITE,
    'source': str(config.GML_PATH.name),
    'layer': config.LAYER,
    'epsg': config.EPSG,
    'anchor_south': list(config.ANCHOR_SOUTH),
    'anchor_north': list(config.ANCHOR_NORTH),
    'feature_counts': counts,
    'component_length_m': {k: round(v.length, 1) for k, v in comps.items()},
    'trimmed_length_m': {k: round(v.length, 1) for k, v in lines.items()},
    'densify_step_m': config.DENSIFY_STEP,
    'sample_step_m': config.SAMPLE_STEP,
    'n_vertices': {k: len(v) for k, v in arrays.items()},
    'separation_m': {
        'min': round(float(sep.min()), 1),
        'p5': round(float(p5), 1),
        'median': round(float(p50), 1),
        'p95': round(float(p95), 1),
        'max': round(float(sep.max()), 1),
    },
    'midline_ratio_median': round(float(np.nanmedian(ratio)), 4),
    'hairpins': hairpins,
}

fn = config.OUT / f'reference_meta_{config.SITE}.json'
with open(fn, 'w') as f:
    json.dump(meta, f, indent=2)
print('wrote', fn.name)

## Retrieval polygon

In [ ]:
# Derived from the shoreline rather than drawn by hand, so downloaded imagery is
# guaranteed to contain the coast. The original hand-drawn box cut off part of the
# coastline near Bridlington

# If this differs from config.POLYGON, update config and re-run
# check_images_available — image counts will change

buf = gpd.GeoSeries([lines['mhw']], crs=config.EPSG).buffer(config.BOX_BUFFER).to_crs(4326)
minx, miny, maxx, maxy = buf.total_bounds

# round outward so the box never shrinks inside the buffered shoreline
minx, miny = np.floor(minx * 100) / 100, np.floor(miny * 100) / 100
maxx, maxy = np.ceil(maxx * 100) / 100, np.ceil(maxy * 100) / 100

polygon = [[[minx, miny], [maxx, miny], [maxx, maxy], [minx, maxy], [minx, miny]]]

print('POLYGON = [[[%.2f, %.2f],' % (minx, miny))
for x, y in polygon[0][1:]:
    print('            [%.2f, %.2f],' % (x, y))
print('           ]]')
print()
print('matches config.POLYGON:', np.allclose(polygon, config.POLYGON))

with open(config.OUT / f'retrieval_polygon_{config.SITE}.json', 'w') as f:
    json.dump({'polygon_wgs84': polygon, 'buffer_m': config.BOX_BUFFER}, f, indent=2)